# Your weather, as a table

Four years of one-minute readings from your own station: **1,946,556 rows**, 2022-10-08 to 2026-09-15.

This notebook is a tour, not a tutorial to read passively. Change the numbers and re-run things.
The point of learning on this data instead of the Titanic is that you already know what the answers
*should* be -- you were standing outside in it.

## 1. Load it

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from vane import Archive, daily, UNITS
from vane.derive import enrich, coverage, wind_vector, bearing

arc = Archive()
arc.root, arc.tz

The first call decodes ~1,400 gzipped NDJSON files and caches Parquet into `cache/`.
Cold: about 10 seconds. Every call after: about 2. Delete `cache/` any time -- the NDJSON is the truth,
Parquet is just a faster copy of it.

In [ ]:
df = arc.samples()          # everything; or arc.samples('2025'), arc.samples('2025-07')
df.shape

In [ ]:
df.head(3)

## 2. What a row is

One reading from the station. Not an hour, not a day -- an instant.

Units are imperial because that is how they were fetched (`TempestClient.swift:52` asks for
`units_temp=f`, `units_wind=mph`, `units_pressure=inhg`). Nothing has been converted.

In [ ]:
pd.DataFrame({'dtype': df.dtypes.astype(str), 'units': pd.Series(UNITS), 'nulls': df.isna().sum()})

`float32` and `int16` are deliberate. In `float64` this table is ~300 MB; as declared it's ~165 MB and
every operation moves half as much memory. Dtype choice **is** the numpy lesson -- it's not housekeeping.

In [ ]:
df.memory_usage(deep=True).sum() / 1e6   # MB

## 3. Coverage comes before any maximum

The archive is emphatic that *no data* and *zero* are different answers
(`DailyRecord.swift:23`). A day whose afternoon is missing understates its own high, and if you
don't filter first, that missing afternoon shows up in your analysis as a cool day.

So: **always look at coverage before you take a max.**

In [ ]:
cov = coverage(df)
cov.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.fill_between(cov.index, cov.fraction, color='#4a7ba7', lw=0)
ax.axhline(0.9, color='#c1121f', ls='--', lw=1, label='90% -- the app\'s reliability gate')
ax.set_ylim(0, 1.05); ax.set_ylabel('fraction of day seen'); ax.legend()
ax.set_title('Coverage. The gaps are outages, not calm weather.')
plt.show()

In [ ]:
whole = cov[cov.fraction >= 0.9].index
print(f'{len(whole)} of {len(cov)} days fully watched')
print(f'worst 5:'); cov.nsmallest(5, 'fraction')

## 4. Resampling: the shape inside a day

This is what minute data buys you and a daily rollup can never give back. `resample` is `groupby`
for time -- the single most useful thing pandas does that numpy doesn't.

In [ ]:
july = arc.samples('2025-07')
hourly = july['airTemp'].resample('1h').mean()
hourly.head()

Now fold every July day onto one 24-hour axis. `minuteOfDay` exists for exactly this -- it's the
same trick `StationSample.minuteOfDay` serves on the Swift side.

In [ ]:
j = enrich(july)
by_minute = j.groupby('minuteOfDay')['airTemp']

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(by_minute.mean().index / 60, by_minute.min(), by_minute.max(),
                alpha=0.25, color='#e07a5f', lw=0, label='range across the month')
ax.plot(by_minute.mean().index / 60, by_minute.mean(), color='#c1121f', lw=2, label='mean')
ax.set_xlabel('hour of day'); ax.set_ylabel('°F'); ax.set_xlim(0, 24)
ax.set_xticks(range(0, 25, 3)); ax.legend()
ax.set_title('July 2025, every day laid over one axis')
plt.show()

**Try:** swap `airTemp` for `windGust` or `humidity`. Wind has a diurnal cycle that peaks in the
afternoon and dies at night -- you can see the boundary layer decoupling after sunset.

## 5. The trap: you cannot average a compass bearing

Wind direction is in degrees, degrees are numbers, `.mean()` will happily give you one. It will be
wrong, and nothing will warn you.

A north wind wobbling between 350° and 10° averages to **180°** -- due south, the exact opposite.
`DailyRecord` has no mean-wind-direction field at all for this reason, and `StationSample.swift`
keeps direction only at sample resolution "where it needs no reduction".

Here it is in your own data:

In [ ]:
d = enrich(df)
g = d.groupby('day')
cmp = pd.DataFrame({'naive': g.windDirection.mean(), 'u': g.u.mean(), 'v': g.v.mean()})
cmp['vector'] = bearing(cmp.u, cmp.v)
cmp['gap'] = ((cmp.naive - cmp.vector + 180) % 360) - 180
cmp.reindex(cmp.gap.abs().sort_values(ascending=False).index).head(5).round(1)

On 2025-10-30 the naive mean says the wind came from **172°** (due south). Resolving into
components and averaging *those* says **351°** -- due north. One of these is 179° wrong.

The fix is to decompose into eastward/northward components, average the components (ordinary
numbers, ordinary mean), then convert back:

In [ ]:
day = d.loc['2025-10-30']
uv = day[['u', 'v']].mean()
print(f'vector mean:  from {bearing(uv.u, uv.v):.0f}°  at {np.hypot(uv.u, uv.v):.2f} mph')
print(f'naive mean:   from {day.windDirection.mean():.0f}°')

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
ax.scatter(np.deg2rad(day.windDirection), day.windAvg, s=2, alpha=0.2, color='#4a7ba7')
ax.set_title('2025-10-30: where the wind actually was')
plt.show()

The vector length is a bonus the naive mean can't offer: a day of wind from every quarter produces
a *short* vector, which tells you the direction meant little. Direction and its own reliability,
in one number.

## 6. Check your work against the app

`daily/` holds the rollups the Swift already computed. That makes it an answer key: every daily
figure you derive has a counterpart to check against.

In [ ]:
roll = daily(arc, 2025)
roll[['tempMin', 'tempMax', 'windGustMax', 'sampleCount', 'expectedSamples']].head(3)

Look at `expectedSamples`: **48**. These rollups were built from a *half-hourly* fetch, while your
sample archive is at one minute. So the two tables disagree -- and the samples should win, because
a 30-minute series cannot see a peak that lasted four minutes. This is exactly what
`SampleMerge.shouldReplace` means by *finer resolution wins*.

In [ ]:
s25 = arc.samples('2025')
mine = s25.groupby(s25.index.strftime('%Y-%m-%d')).agg(
    myTemp=('airTemp', 'max'), myGust=('windGust', 'max'), n=('airTemp', 'size'))

j = roll.join(mine).query('fraction >= 0.9 and n >= 1300')
gap = (j.myTemp - j.tempMax).astype(float)
print(f'comparable days: {len(j)}')
print(f'minute data finds a HIGHER high on {(gap > 0).mean():.0%} of days')
print(f'  median {gap.median():+.2f} °F, worst {gap.max():+.1f} °F')

89%. Not a bug in either table -- a measured demonstration that resolution changes the answer.
It's also a warning: **never compare an extreme from one resolution against an extreme from another.**
If you ever mix this archive with data sampled differently, that comparison is the first thing to break.

## 7. Where to go next

Things this data can answer that a tutorial dataset can't:

- **Pressure and storms.** `seaLevelPressure.diff()` over a rolling 3-hour window. Do the sharp
  drops line up with `strikeCount > 0`? You were there -- you can check the answer against memory.
- **Dew point vs. humidity.** `dewPoint` is already computed per reading. Plot both across a summer
  day and watch humidity swing wildly while dew point sits nearly flat. One of them is a property
  of the air mass; the other is a property of the thermometer.
- **Growing degree days / chill hours.** The rollup has `chillHours` and `chillUnitsUtah`.
  Recompute them from minutes and see how much the half-hourly version missed.
- **Your own microclimate.** Group `airTemp` by `windDirection` sector. Does a south wind run warmer
  at your station than a north wind at the same time of year?

Two more tables are sitting on disk, unread by this notebook:

- `forecasts/` -- what was *predicted* for each day, as a log of revisions. Joined to what actually
  happened, that's a labeled supervised-learning problem with a reference implementation in
  `ForecastAccuracy.swift` to check against.
- `upper-air/` -- soundings.

Ask and I'll add readers for either.